# Học máy Tastoria — đọc kết quả

Sổ này chỉ **đọc** các tệp JSON do các lệnh thí nghiệm ghi ra (không tính lại số). Chạy lại thí nghiệm:

```
CUDA_VISIBLE_DEVICES= uv run --package tastoria-ml python -m tastoria_ml.recsys.foodcom
uv run --package tastoria-ml python -m tastoria_ml.recsys.survey simulate
uv run --package tastoria-ml python -m tastoria_ml.composer_eval.run
uv run --package tastoria-ml python -m tastoria_ml.substitution.run
uv run --package tastoria-ml python -m tastoria_ml.recsys.figures
```

In [ ]:
import json, pathlib
import pandas as pd
P = pathlib.Path('../../data/processed')
load = lambda rel: json.loads((P / rel).read_text(encoding='utf-8'))

## 1. Food.com (test, trung bình ± sd qua seed)

In [ ]:
t = load('recsys/test.json')
pd.DataFrame({m: {k: f"{v[k]['mean']:.4f} ± {v[k]['sd']:.4f}" for k in ('hr@10','recall@10','ndcg@10')} for m, v in t['models'].items()}).T

In [ ]:
pd.DataFrame({m: v['ndcg@10'] for m, v in t['comparisons']['per_model'].items()}).T

## 2. Khảo sát — MÔ PHỎNG (chỉ kiểm đường ống)

In [ ]:
s = load('recsys/survey_sim.json')
pd.concat({sc: pd.DataFrame({m: {k: v[k]['mean'] for k in ('ndcg@5','auc')} for m, v in r['models'].items()}).T for sc, r in s['scenarios'].items()}, axis=1)

## 3. Bộ ghép mâm

In [ ]:
c = load('composer_eval/summary.json')
c['chosen'], c['test_chosen_row']

## 4. Thay thế nguyên liệu (đồng thuận với bảng tra, KHÔNG phải độ chính xác)

In [ ]:
u = load('substitution/results.json')
pd.DataFrame({m: {k: (v[k]['mean'] if isinstance(v[k], dict) else v[k]) for k in ('agreement@1','agreement@3','mrr','answered')} for m, v in u['test'].items()}).T